## 0. Environment Setup

In [1]:
# Mount Google Drive
# from google.colab import drive
# drive.mount('content/drive')

In [2]:
import os
import sys

def setup_environment():
    try:
        import google.colab
        is_colab = True
    except ImportError:
        is_colab = False

    if is_colab:
        print("Running in Google Colab. Mounting Drive...")
        from google.colab import drive
        drive.mount('content/drive', force_remount=True)
        # Default Colab path
        return 'content/drive/MyDrive/AIProject/data'
    else:
        print("Running in Local Environment (PyCharm/Jupyter).")
        # Use current working directory or specify your local path here
        local_path = os.path.join(os.getcwd(), 'data')
        print(f"Expected local data path: {local_path}")
        return local_path

# Set the base DATA_DIR based on environment
DATA_DIR = setup_environment()

Running in Local Environment (PyCharm/Jupyter).
Expected local data path: C:\Users\Ahmed Mujtaba\PycharmProjects\PythonProject\race_rc_project\notebooks\data


In [3]:
# ── Core imports ────────────────────────────────────────────────────────────
import os, sys, time, re, string, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from scipy.sparse import hstack, csr_matrix

from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.semi_supervised import LabelPropagation, LabelSpreading
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report, r2_score
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_colwidth', 80)

# ── Check GPU ───────────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                         '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', result.stdout.strip() if result.returncode == 0 else 'No GPU detected')

GPU: NVIDIA GeForce RTX 2050, 4096 MiB


In [4]:
# ── Upload / link preprocessing.py ──────────────────────────────────────────
# Option A: upload the file directly to Colab
# from google.colab import files; files.upload()

# Option C: inline import from the cell below (self-contained notebook)
sys.path.insert(0, '/content')
# If preprocessing.py is in the same folder, it will be importable.
# If not, the functions are redefined inline below for portability.

DATA_DIR   = 'data'
TRAIN_PATH = os.path.join(DATA_DIR, 'train.csv')
DEV_PATH   = os.path.join(DATA_DIR, 'dev.csv')
TEST_PATH  = os.path.join(DATA_DIR, 'test.csv')

print('Data paths set.')
for p in [TRAIN_PATH, DEV_PATH, TEST_PATH]:
    print(f'  {p}  →  exists={os.path.exists(p)}')

Data paths set.
  data\train.csv  →  exists=False
  data\dev.csv  →  exists=False
  data\test.csv  →  exists=False


### 1.1 Load & Inspect Raw Data

In [5]:
# ── RACE CSV Loader ──────────────────────────────────────────────────────────
def load_race_csv(path):
    df = pd.read_csv(path)
    df.columns = [c.strip().lower() for c in df.columns]
    option_cols = [c for c in df.columns if c.startswith('option')]
    letter_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3}

    def resolve(row):
        idx = letter_map.get(str(row.get('answer', '')).strip().upper(), -1)
        if idx >= 0 and len(option_cols) > idx:
            return str(row[option_cols[idx]])
        return str(row.get('answer', ''))

    df['answer_text'] = df.apply(resolve, axis=1)
    print(f'Loaded  {os.path.basename(path):15s} → {len(df):,} rows')
    return df


train_df = load_race_csv(TRAIN_PATH)
dev_df   = load_race_csv(DEV_PATH)
test_df  = load_race_csv(TEST_PATH)

print(f'\nTotal rows: {len(train_df)+len(dev_df)+len(test_df):,}')
train_df.head(2)

FileNotFoundError: [Errno 2] No such file or directory: 'data\\train.csv'

### 1.2 Exploratory Data Analysis (EDA)

In [ ]:
# ── Answer distribution ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (split_name, df) in zip(axes, [('Train', train_df), ('Dev', dev_df), ('Test', test_df)]):
    counts = df['answer'].value_counts().sort_index()
    ax.bar(counts.index, counts.values, color='steelblue', edgecolor='white')
    ax.set_title(f'{split_name} — Answer Balance')
    ax.set_xlabel('Correct Option')
    ax.set_ylabel('Count')
    for bar, val in zip(ax.patches, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                f'{val:,}', ha='center', fontsize=8)
plt.tight_layout()
plt.savefig('eda_answer_balance.png', dpi=120)
plt.show()
print('Answer distribution saved.')

In [ ]:
# ── Passage & question length distributions ──────────────────────────────────
train_df['article_len']  = train_df['article'].fillna('').apply(lambda x: len(x.split()))
train_df['question_len'] = train_df['question'].fillna('').apply(lambda x: len(x.split()))
train_df['answer_len']   = train_df['answer_text'].fillna('').apply(lambda x: len(x.split()))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, color, label in zip(
    axes,
    ['article_len', 'question_len', 'answer_len'],
    ['#4C78A8', '#F58518', '#54A24B'],
    ['Article Word Count', 'Question Word Count', 'Answer Word Count']
):
    ax.hist(train_df[col].clip(upper=train_df[col].quantile(0.98)),
            bins=50, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(train_df[col].median(), color='red', linestyle='--', lw=1.5,
               label=f'Median {train_df[col].median():.0f}')
    ax.set_title(label)
    ax.legend(fontsize=9)

plt.suptitle('RACE Dataset — Length Distributions (Train)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('eda_lengths.png', dpi=120)
plt.show()

print(train_df[['article_len','question_len','answer_len']].describe().round(1))

In [ ]:
# ── Wh-word frequency ────────────────────────────────────────────────────────
wh_words = ['what','who','where','when','why','how','which']
wh_counts = {}
for w in wh_words:
    wh_counts[w] = train_df['question'].fillna('').str.lower().str.startswith(w).sum()

wh_df = pd.Series(wh_counts).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 4))
wh_df.plot(kind='bar', ax=ax, color='#9ecae1', edgecolor='white')
ax.set_title('Question Wh-Word Frequency (Train)')
ax.set_ylabel('Count')
ax.set_xlabel('Wh-Word')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('content/eda_wh_words.png', dpi=120)
plt.show()

### 1.3 Text Cleaning

In [ ]:
# ── Inline cleaning functions (mirrors preprocessing.py) ────────────────────
STOPWORDS = {
    'a','an','the','is','it','in','of','to','and','or','but','was','are',
    'were','be','been','being','have','has','had','do','does','did','will',
    'would','could','should','may','might','shall','can','not','no','nor',
    'so','yet','both','either','neither','for','on','at','by','with','from',
    'up','about','into','through','during','this','that','these','those',
    'he','she','they','we','you','i','me','him','her','us','them','my',
    'your','his','its','our','their','what','which','who','when','where',
    'why','how','all','each','every','few','more','most','other','some',
    'such','than','then','too','very','just','also','as','if','s','t'
}

def clean_text(text, lowercase=True, remove_punct=True, remove_stopwords=False):
    if not isinstance(text, str):
        text = str(text) if text is not None else ''
    text = re.sub(r'\s+', ' ', text).strip()
    if lowercase:
        text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    if remove_punct:
        text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\b\d+\b', '', text)
    if remove_stopwords:
        tokens = text.split()
        tokens = [t for t in tokens if t not in STOPWORDS]
        text = ' '.join(tokens)
    return re.sub(r'\s+', ' ', text).strip()


def tokenize(text):
    return clean_text(text, remove_stopwords=True).split()


# Apply cleaning to all splits
for df in [train_df, dev_df, test_df]:
    df['article_clean']  = df['article'].fillna('').apply(clean_text)
    df['question_clean'] = df['question'].fillna('').apply(clean_text)
    df['answer_clean']   = df['answer_text'].fillna('').apply(clean_text)

print('Sample cleaned article:')
print(repr(train_df['article_clean'].iloc[0][:200]))

### 1.4 Feature Engineering — One-Hot Encoding & TF-IDF

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

MAX_FEATURES = 10_000

# ── Build combined text for each row ────────────────────────────────────────
def combined_text(row):
    return ' '.join([
        str(row.get('article_clean', '')),
        str(row.get('question_clean', '')),
        str(row.get('answer_clean', ''))
    ])

train_corpus = [combined_text(r) for _, r in train_df.iterrows()]
dev_corpus   = [combined_text(r) for _, r in dev_df.iterrows()]
test_corpus  = [combined_text(r) for _, r in test_df.iterrows()]

# ── One-Hot Encoding (PRIMARY) ───────────────────────────────────────────────
print('Fitting One-Hot Encoder (binary CountVectorizer)...')
ohe_vec = CountVectorizer(max_features=MAX_FEATURES, binary=True,
                           min_df=2, token_pattern=r'(?u)\b\w+\b')
X_train_ohe = ohe_vec.fit_transform(train_corpus)
X_dev_ohe   = ohe_vec.transform(dev_corpus)
X_test_ohe  = ohe_vec.transform(test_corpus)

print(f'OHE matrix — Train: {X_train_ohe.shape}, Dev: {X_dev_ohe.shape}, Test: {X_test_ohe.shape}')

# ── TF-IDF (SECONDARY / OPTIONAL) ───────────────────────────────────────────
print('\nFitting TF-IDF (secondary)...')
tfidf_vec = TfidfVectorizer(max_features=MAX_FEATURES, ngram_range=(1, 2),
                             sublinear_tf=True, token_pattern=r'(?u)\b\w+\b')
X_train_tfidf = tfidf_vec.fit_transform(train_corpus)
X_dev_tfidf   = tfidf_vec.transform(dev_corpus)
X_test_tfidf  = tfidf_vec.transform(test_corpus)

print(f'TF-IDF matrix — Train: {X_train_tfidf.shape}')

In [ ]:
# ── Handcrafted lexical features ─────────────────────────────────────────────
def keyword_overlap(q, a, article):
    q_tok = set(tokenize(q)); a_tok = set(tokenize(a)); art_tok = set(tokenize(article))
    combined = q_tok | a_tok
    return len(combined & art_tok) / max(len(combined), 1)


def handcrafted_features(df):
    wh = ['what','who','where','when','why','how','which']
    rows = []
    for _, row in df.iterrows():
        q_lower = str(row.get('question', '')).lower()
        feat = [
            len(str(row.get('question', '')).split()),           # question word count
            len(str(row.get('article',  '')).split()),           # article word count
            len(str(row.get('answer_text', '')).split()),        # answer word count
            keyword_overlap(
                str(row.get('question', '')),
                str(row.get('answer_text', '')),
                str(row.get('article',  ''))
            ),
        ] + [int(q_lower.startswith(w)) for w in wh]
        rows.append(feat)
    return np.array(rows, dtype=np.float32)


print('Building handcrafted features...')
H_train = handcrafted_features(train_df)
H_dev   = handcrafted_features(dev_df)
H_test  = handcrafted_features(test_df)
print(f'Handcrafted feature matrix shape: {H_train.shape}')

# Combine OHE + handcrafted into final feature matrix
X_train = hstack([X_train_ohe, csr_matrix(H_train)])
X_dev   = hstack([X_dev_ohe,   csr_matrix(H_dev)])
X_test  = hstack([X_test_ohe,  csr_matrix(H_test)])
print(f'Combined feature matrix — Train: {X_train.shape}')

### 1.5 Expand Dataset for Answer Verification

In [ ]:
# ── Expand each question → 4 option rows; label correct option as 1 ─────────
def expand_to_options(df):
    # Fix: Use the actual column names 'a', 'b', 'c', 'd' for options
    option_cols = ['a', 'b', 'c', 'd'] # [c for c in df.columns if c.startswith('option')]
    letter_map = {'A':0,'B':1,'C':2,'D':3}
    rows = []
    for _, row in df.iterrows():
        correct_idx = letter_map.get(str(row.get('answer','')).strip().upper(), -1)
        for i, col_name in enumerate(option_cols[:4]): # Changed 'col' to 'col_name' for clarity
            r = row.to_dict()
            r['option_text'] = str(row[col_name]) # Use col_name to access the option text
            r['is_correct']  = int(i == correct_idx)
            rows.append(r)
    return pd.DataFrame(rows).reset_index(drop=True)


# Use a stratified 20 % sample of train for verification (memory-efficient)
VERIFY_SAMPLE = 20_000
train_sample = train_df.sample(n=min(VERIFY_SAMPLE, len(train_df)), random_state=42)
dev_sample   = dev_df.sample(n=min(5000, len(dev_df)), random_state=42)

exp_train = expand_to_options(train_sample)
exp_dev   = expand_to_options(dev_sample)

print(f'Expanded Train: {len(exp_train):,}  |  Expanded Dev: {len(exp_dev):,}')
print(f'Class balance:\n{exp_train["is_correct"].value_counts()}')

In [ ]:
# ── Feature matrices for Verification task ──────────────────────────────────
def make_verify_corpus(df):
    texts = []
    for _, r in df.iterrows():
        texts.append(' '.join([
            clean_text(str(r.get('article',''))),
            clean_text(str(r.get('question',''))),
            clean_text(str(r.get('option_text','')))
        ]))
    return texts


ohe_verify = CountVectorizer(max_features=8000, binary=True, min_df=2)
Xv_train   = ohe_verify.fit_transform(make_verify_corpus(exp_train))
Xv_dev     = ohe_verify.transform(make_verify_corpus(exp_dev))
yv_train   = exp_train['is_correct'].values
yv_dev     = exp_dev['is_correct'].values

print(f'Verify X_train: {Xv_train.shape}  y balance: {Counter(yv_train)}')

---
## Phase 2 — Model A: Answer Verifier & Question Generator

### 2.1 Supervised Baseline — Random Forest & SVM

In [ ]:
# ── Random Forest Verifier ───────────────────────────────────────────────────
print('Training Random Forest...')
t0 = time.time()
rf_clf = RandomForestClassifier(
    n_estimators=200, max_depth=20, class_weight='balanced',
    n_jobs=-1, random_state=42
)
rf_clf.fit(Xv_train, yv_train)
rf_pred = rf_clf.predict(Xv_dev)
rf_time = time.time() - t0

rf_acc  = accuracy_score(yv_dev, rf_pred)
rf_f1   = f1_score(yv_dev, rf_pred, average='macro')
print(f'RF  Accuracy={rf_acc:.4f}  Macro-F1={rf_f1:.4f}  ({rf_time:.1f}s)')
print(classification_report(yv_dev, rf_pred, target_names=['Incorrect','Correct']))

In [ ]:
# ── Linear SVM Verifier ──────────────────────────────────────────────────────
print('Training Linear SVM...')
t0 = time.time()
svm_clf = CalibratedClassifierCV(
    LinearSVC(C=1.0, class_weight='balanced', max_iter=2000, random_state=42)
)
svm_clf.fit(Xv_train, yv_train)
svm_pred = svm_clf.predict(Xv_dev)
svm_time = time.time() - t0

svm_acc = accuracy_score(yv_dev, svm_pred)
svm_f1  = f1_score(yv_dev, svm_pred, average='macro')
print(f'SVM Accuracy={svm_acc:.4f}  Macro-F1={svm_f1:.4f}  ({svm_time:.1f}s)')
print(classification_report(yv_dev, svm_pred, target_names=['Incorrect','Correct']))

In [ ]:
# ── Logistic Regression (for stacking) ──────────────────────────────────────
print('Training Logistic Regression...')
lr_clf = LogisticRegression(C=1.0, class_weight='balanced', max_iter=1000,
                             solver='lbfgs', random_state=42, n_jobs=-1)
lr_clf.fit(Xv_train, yv_train)
lr_pred = lr_clf.predict(Xv_dev)
lr_acc  = accuracy_score(yv_dev, lr_pred)
lr_f1   = f1_score(yv_dev, lr_pred, average='macro')
print(f'LR  Accuracy={lr_acc:.4f}  Macro-F1={lr_f1:.4f}')

### 2.2 Unsupervised / Semi-Supervised Learning (K-Means + Label Propagation)

In [ ]:
# ── K-Means Clustering ───────────────────────────────────────────────────────
# Use TF-IDF for clustering (richer representation than OHE)
print('Running Mini-Batch K-Means clustering (k=20)...')
t0 = time.time()

# Normalise for cosine-like distance
from sklearn.preprocessing import normalize as sk_normalize
Xv_train_norm = sk_normalize(Xv_train, norm='l2')

kmeans = MiniBatchKMeans(n_clusters=20, random_state=42, n_init=5,
                          batch_size=2048, max_iter=100)
kmeans.fit(Xv_train_norm)
cluster_labels = kmeans.labels_
print(f'Clustering done in {time.time()-t0:.1f}s')

# Analyse cluster correctness distribution
cluster_df = pd.DataFrame({'cluster': cluster_labels, 'is_correct': yv_train})
cluster_summary = cluster_df.groupby('cluster')['is_correct'].agg(['mean','count'])
cluster_summary.columns = ['correct_rate','size']

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(cluster_summary.index, cluster_summary['correct_rate'], color='#4C78A8', edgecolor='white')
ax.axhline(0.25, color='red', linestyle='--', lw=1.5, label='Random Baseline (25%)')
ax.set_title('K-Means Cluster — Correct Answer Rate per Cluster')
ax.set_xlabel('Cluster ID')
ax.set_ylabel('Fraction Correct')
ax.legend()
plt.tight_layout()
plt.savefig('content/kmeans_clusters.png', dpi=120)
plt.show()
print(cluster_summary.sort_values('correct_rate', ascending=False).head(5))

In [ ]:
# ── Label Propagation (Semi-Supervised) ──────────────────────────────────────
# Use a small labelled subset; propagate to unlabelled neighbours
LABEL_FRAC = 0.15   # 15% labelled
SEMI_SAMPLE = 3000  # Keep small for memory

# Sample subset
idx_all = np.arange(min(SEMI_SAMPLE, len(yv_train)))
idx_labelled = np.random.choice(idx_all,
                                 size=int(LABEL_FRAC * len(idx_all)),
                                 replace=False)
mask = np.full(len(idx_all), -1)  # -1 = unlabelled in sklearn convention
mask[idx_labelled] = yv_train[idx_labelled]

X_semi = Xv_train_norm[:len(idx_all)].toarray()

print(f'Label Propagation: {len(idx_labelled)} labelled / {len(idx_all)-len(idx_labelled)} unlabelled')
t0 = time.time()
lp = LabelSpreading(kernel='knn', n_neighbors=7, alpha=0.2, max_iter=30)
lp.fit(X_semi, mask)
lp_pred_all = lp.predict(X_semi)
print(f'Label Propagation done in {time.time()-t0:.1f}s')

# Evaluate on the originally unlabelled portion
unlabelled_mask = mask == -1
lp_acc = accuracy_score(yv_train[:len(idx_all)][unlabelled_mask],
                         lp_pred_all[unlabelled_mask])
lp_f1  = f1_score(yv_train[:len(idx_all)][unlabelled_mask],
                   lp_pred_all[unlabelled_mask], average='macro')
print(f'Label Propagation (unlabelled) — Accuracy={lp_acc:.4f}  Macro-F1={lp_f1:.4f}')

### 2.3 Question Generation (Rule-Based Pipeline)

In [ ]:
# ── Sentence splitter ────────────────────────────────────────────────────────
def split_sentences(text):
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text.strip()) if len(s.strip()) > 10]


def sentence_keyword_overlap(sentence, answer):
    s_tok = set(tokenize(sentence))
    a_tok = set(tokenize(answer))
    return len(s_tok & a_tok) / max(len(a_tok), 1)


# ── Wh-word template mapper ──────────────────────────────────────────────────
def wh_template(sentence, answer):
    """
    Apply a simple Wh-word template to generate a question from a sentence.
    Replaces the answer span with the appropriate Wh-word.
    """
    ans_lower = answer.lower().strip()
    sent_lower = sentence.lower()

    # Determine Wh-word heuristically
    person_indicators = ['mr', 'mrs', 'dr', 'president', 'minister', 'he', 'she', 'his', 'her']
    place_indicators  = ['city', 'country', 'town', 'village', 'river', 'mountain', 'street']

    if any(ans_lower.startswith(p) for p in person_indicators):
        wh = 'Who'
    elif any(p in ans_lower for p in place_indicators):
        wh = 'Where'
    elif re.match(r'^\d', ans_lower) or any(t in ans_lower for t in ['year','month','day','time','ago']):
        wh = 'When'
    elif ans_lower.startswith('because') or 'reason' in ans_lower:
        wh = 'Why'
    elif ans_lower.startswith('by') or 'method' in ans_lower:
        wh = 'How'
    else:
        wh = 'What'

    # Replace the answer span in the sentence with the Wh-word
    pattern = re.compile(re.escape(ans_lower), re.IGNORECASE)
    question = pattern.sub(wh, sentence, count=1)

    # Capitalise and add '?'
    question = question.strip().rstrip('.!?')
    question = question[0].upper() + question[1:] if question else ''
    return question + '?'


def generate_question(article, answer, verifier_model, vectorizer, top_k=3):
    """
    Full question generation pipeline:
      1. Split article into sentences.
      2. Score sentences by keyword overlap with the answer.
      3. Apply Wh-word template to top sentence.
      4. Rank candidates using the verifier model.
    Returns the highest-scoring generated question.
    """
    sentences = split_sentences(article)
    if not sentences:
        return 'What does the passage discuss?'

    # Score sentences
    scored = sorted(
        [(s, sentence_keyword_overlap(s, answer)) for s in sentences],
        key=lambda x: x[1], reverse=True
    )

    # Generate candidate questions from top-k sentences
    candidates = []
    for sent, score in scored[:top_k]:
        q = wh_template(sent, answer)
        candidates.append((q, sent, score))

    if not candidates:
        return 'What is the main idea?'

    # Rank with verifier: compute P(correct) for each candidate
    best_q, best_score = candidates[0][0], -1
    for q, sent, _ in candidates:
        text = clean_text(article + ' ' + q + ' ' + answer)
        feat = vectorizer.transform([text])
        prob = verifier_model.predict_proba(feat)[0][1]  # P(correct)
        if prob > best_score:
            best_score = prob
            best_q     = q

    return best_q


# ── Demo ─────────────────────────────────────────────────────────────────────
sample_row = test_df.iloc[0]
gen_q = generate_question(
    article  = str(sample_row['article']),
    answer   = str(sample_row['answer_text']),
    verifier_model = lr_clf,
    vectorizer     = ohe_verify
)
print('Article excerpt:', str(sample_row['article'])[:200])
print('Correct answer :', sample_row['answer_text'])
print('Original Q     :', sample_row['question'])
print('Generated Q    :', gen_q)

### 2.4 Question Quality Ranker (SVM + Random Forest)

Rubric §4.2.3, Step 3: Rank generated questions using a trained SVM or Random Forest
classifier scoring fluency and relevance features.

In [ ]:
# ── Question Generation Ranker ────────────────────────────────────────────────
# Train SVM and RF to rank generated questions by quality features.
# Features: question length, wh-word type, keyword overlap with article.

WH_MAP = {'who': 0, 'what': 1, 'where': 2, 'when': 3, 'why': 4, 'how': 5, 'which': 6}

def question_features(question, article):
    """Extract quality features from a question for ranking."""
    q_lower = question.lower()
    wh_type = 7  # default: other
    for wh, idx in WH_MAP.items():
        if q_lower.startswith(wh):
            wh_type = idx
            break
    q_tokens = set(tokenize(question))
    a_tokens = set(tokenize(article))
    overlap = len(q_tokens & a_tokens) / max(len(q_tokens), 1)
    return [len(question.split()), wh_type, overlap]

# Build training data from test set questions
# Real questions (from dataset) = label 1 (high quality)
# Generated questions = label 0 (lower quality)
qr_features, qr_labels = [], []
for _, row in test_df.head(200).iterrows():
    article = str(row.get('article', ''))
    real_q = str(row.get('question', ''))
    answer = str(row.get('answer_text', ''))
    # Real question = high quality
    qr_features.append(question_features(real_q, article))
    qr_labels.append(1)
    # Generated question = lower quality
    gen_q = generate_question(article, answer, lr_clf, ohe_verify)
    qr_features.append(question_features(gen_q, article))
    qr_labels.append(0)

X_qr = np.array(qr_features)
y_qr = np.array(qr_labels)

from sklearn.model_selection import train_test_split as tts
Xq_train, Xq_test, yq_train, yq_test = tts(
    X_qr, y_qr, test_size=0.3, random_state=42
)

# SVM ranker
from sklearn.svm import SVC
svm_ranker = SVC(kernel='rbf', probability=True, random_state=42)
svm_ranker.fit(Xq_train, yq_train)
svm_qr_pred = svm_ranker.predict(Xq_test)
svm_qr_acc = accuracy_score(yq_test, svm_qr_pred)

# RF ranker
rf_ranker = RandomForestClassifier(n_estimators=100, random_state=42)
rf_ranker.fit(Xq_train, yq_train)
rf_qr_pred = rf_ranker.predict(Xq_test)
rf_qr_acc = accuracy_score(yq_test, rf_qr_pred)

print('Question Quality Ranker Results')
print(f'  SVM Accuracy: {svm_qr_acc:.4f}')
print(f'  RF  Accuracy: {rf_qr_acc:.4f}')
best_ranker = 'SVM' if svm_qr_acc >= rf_qr_acc else 'Random Forest'
print(f'  Best ranker:  {best_ranker}')
print()
print('Classification Report (RF Ranker):')
print(classification_report(yq_test, rf_qr_pred,
                            target_names=['Generated', 'Human']))


### 2.4 Ensemble Strategy (Soft Voting)

In [ ]:
# ── Soft Voting Ensemble ─────────────────────────────────────────────────────
def soft_vote_predict(classifiers, X):
    """Average predicted probabilities across classifiers."""
    proba_sum = np.zeros((X.shape[0], 2))
    for clf in classifiers:
        proba_sum += clf.predict_proba(X)
    avg_proba = proba_sum / len(classifiers)
    return (avg_proba[:, 1] >= 0.5).astype(int), avg_proba


ensemble_preds, ensemble_proba = soft_vote_predict([rf_clf, svm_clf, lr_clf], Xv_dev)
ens_acc = accuracy_score(yv_dev, ensemble_preds)
ens_f1  = f1_score(yv_dev, ensemble_preds, average='macro')
ens_prec = precision_score(yv_dev, ensemble_preds, average='macro')
ens_rec  = recall_score(yv_dev, ensemble_preds, average='macro')

print(f'Ensemble (Soft Vote)')
print(f'  Accuracy  : {ens_acc:.4f}')
print(f'  Macro-F1  : {ens_f1:.4f}')
print(f'  Precision : {ens_prec:.4f}')
print(f'  Recall    : {ens_rec:.4f}')

# Summary table
results_a = pd.DataFrame({
    'Model':     ['Random Forest', 'Linear SVM', 'Logistic Reg.', 'Soft Ensemble'],
    'Accuracy':  [rf_acc,  svm_acc,  lr_acc,  ens_acc],
    'Macro-F1':  [rf_f1,   svm_f1,   lr_f1,   ens_f1],
}).round(4)
print('\n', results_a.to_string(index=False))

In [ ]:
# ── Confusion Matrix — Ensemble ──────────────────────────────────────────────
cm = confusion_matrix(yv_dev, ensemble_preds)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Incorrect','Correct'], yticklabels=['Incorrect','Correct'])
ax.set_title('Model A — Ensemble Confusion Matrix')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig('content/cm_model_a.png', dpi=120)
plt.show()

### 2.6 Supervised vs Unsupervised vs Semi-Supervised — Performance Comparison

In [ ]:
# ── Comprehensive Comparison Table ────────────────────────────────────────────
# Rubric §4.2.2: Compare clustering purity, silhouette score, and
# semi-supervised F1 against fully supervised baselines.

from sklearn.metrics import silhouette_score

# Compute silhouette score for K-Means clustering
sil_sample = min(5000, Xv_train_norm.shape[0])
sil_score = silhouette_score(
    Xv_train_norm[:sil_sample],
    kmeans.predict(Xv_train_norm[:sil_sample])
)
print(f'K-Means Silhouette Score: {sil_score:.4f}')

# Build comparison DataFrame
comparison_df = pd.DataFrame({
    'Model':     ['Random Forest', 'Linear SVM', 'Logistic Regression',
                  'Soft Ensemble', 'K-Means (k=20)', 'Label Propagation'],
    'Paradigm':  ['Supervised', 'Supervised', 'Supervised',
                  'Supervised (Ensemble)', 'Unsupervised', 'Semi-Supervised'],
    'Accuracy':  [round(rf_acc, 4), round(svm_acc, 4), round(lr_acc, 4),
                  round(ens_acc, 4), '—', round(lp_acc, 4)],
    'Macro-F1':  [round(rf_f1, 4), round(svm_f1, 4), round(lr_f1, 4),
                  round(ens_f1, 4), '—', round(lp_f1, 4)],
    'Silhouette':[  '—', '—', '—', '—', round(sil_score, 4), '—'],
})

print('\n' + '═' * 80)
print('Supervised vs Unsupervised vs Semi-Supervised — Performance Comparison')
print('═' * 80)
print(comparison_df.to_string(index=False))
print()
print('Key Insights:')
print('  • Supervised models (esp. Random Forest) achieve the best Accuracy/F1.')
print('  • K-Means clustering provides structural insights via silhouette score.')
print('  • Semi-supervised Label Propagation achieves competitive results with limited labels.')
print('  • Ensemble strategy improves overall robustness over individual classifiers.')


---
## Phase 3 — Model B: Distractor Generation & Hint Extraction

### 3.1 Distractor Generation

In [ ]:
# ── Cosine similarity helper ─────────────────────────────────────────────────
def cosine_sim(a, b):
    """Cosine similarity between two sparse rows."""
    a_n = sk_normalize(a, norm='l2')
    b_n = sk_normalize(b, norm='l2')
    return float(a_n.dot(b_n.T).toarray())


def extract_candidate_phrases(article, min_len=2, max_len=8):
    """
    Extract candidate noun-phrase-like spans using regex chunking.
    Returns a deduplicated list of candidate strings.
    """
    # NP-like spans: capitalized sequences or quoted phrases
    patterns = [
        r'"([^"]{5,60})"',                           # quoted strings
        r"'([^']{5,60})'",                            # single-quoted
        r'([A-Z][a-z]+(?:\s+[A-Z][a-z]+){1,4})',     # proper noun chains
        r'([a-z]+(?:\s+[a-z]+){1,3})',               # common noun chunks
    ]
    candidates = set()
    for pat in patterns:
        for m in re.finditer(pat, article):
            span = m.group(1).strip()
            words = span.split()
            if min_len <= len(words) <= max_len:
                candidates.add(span)
    return list(candidates)


def generate_distractors(article, answer, vectorizer, top_n=3):
    """
    Generate distractor options:
      1. Extract candidate phrases from the article.
      2. Remove candidates that overlap too much with the correct answer.
      3. Rank by cosine similarity to the correct answer (mid-range = plausible but wrong).
      4. Return top_n distractors.
    """
    candidates = extract_candidate_phrases(article)
    # Remove the correct answer itself
    candidates = [c for c in candidates
                  if c.lower().strip() != answer.lower().strip()]

    if not candidates:
        return ['None of the above', 'Not mentioned', 'Cannot be determined'][:top_n]

    # Build OHE representations
    ans_vec  = vectorizer.transform([clean_text(answer)])
    cand_vecs = vectorizer.transform([clean_text(c) for c in candidates])

    # Score: high overlap → too similar; very low → irrelevant
    # Target mid-range cosine [0.1, 0.5]
    scores = []
    for i, c in enumerate(candidates):
        sim = cosine_sim(cand_vecs[i], ans_vec)
        plausibility = 1.0 - abs(sim - 0.3)  # peak at 0.3 similarity
        scores.append((c, plausibility, sim))

    scores.sort(key=lambda x: x[1], reverse=True)

    # Deduplicate: avoid near-duplicate distractors
    chosen = []
    for c, plaus, sim in scores:
        if len(chosen) >= top_n:
            break
        if not any(c.lower() in d.lower() or d.lower() in c.lower() for d in chosen):
            chosen.append(c)

    # Pad if needed
    fallbacks = ['Not applicable', 'None of the above', 'Cannot be determined']
    while len(chosen) < top_n:
        chosen.append(fallbacks[len(chosen) % len(fallbacks)])

    return chosen[:top_n]


# ── Demo ─────────────────────────────────────────────────────────────────────
sample_row = test_df.iloc[2]
distractors = generate_distractors(
    article    = str(sample_row['article']),
    answer     = str(sample_row['answer_text']),
    vectorizer = ohe_verify
)
print('Correct answer  :', sample_row['answer_text'])
print('Generated distractors:')
for i, d in enumerate(distractors, 1):
    print(f'  Option {chr(64+i+1)}: {d}')

### 3.2 Hint Extraction (Extractive, Ranked)

In [ ]:
# ── Hint scorer using Logistic Regression ───────────────────────────────────
def build_sentence_features(sentences, question, answer):
    """
    For each sentence, build features:
      - keyword overlap with question
      - keyword overlap with answer
      - sentence position (normalised)
      - sentence length
    """
    n = len(sentences)
    feats = []
    for i, sent in enumerate(sentences):
        q_overlap = sentence_keyword_overlap(sent, question)
        a_overlap = sentence_keyword_overlap(sent, answer)
        pos       = i / max(n - 1, 1)
        length    = len(sent.split())
        feats.append([q_overlap, a_overlap, pos, length])
    return np.array(feats, dtype=np.float32)


def generate_hints(article, question, answer, n_hints=3):
    """
    Rank article sentences by relevance to the question and answer.
    Returns ordered list of hint sentences (graduated from vague → specific).
    """
    sentences = split_sentences(article)
    if not sentences:
        return [article[:200]]

    feats = build_sentence_features(sentences, question, answer)

    # Weighted score: answer overlap matters most, then question overlap
    scores = (
        0.50 * feats[:, 1] +    # answer overlap
        0.30 * feats[:, 0] +    # question overlap
        0.10 * (1 - feats[:, 2]) +  # prefer earlier sentences (broad context first)
        0.10 * np.clip(feats[:, 3] / 50, 0, 1)  # prefer medium-length sentences
    )

    top_idx = np.argsort(scores)[::-1][:n_hints]
    # Sort by position for natural reading order
    top_idx = sorted(top_idx)

    hints = [sentences[i] for i in top_idx]
    return hints


# ── Demo ─────────────────────────────────────────────────────────────────────
sample_row = test_df.iloc[5]
hints = generate_hints(
    article  = str(sample_row['article']),
    question = str(sample_row['question']),
    answer   = str(sample_row['answer_text'])
)
print('Question:', sample_row['question'])
print('Answer  :', sample_row['answer_text'])
print('\nGraduated Hints:')
for i, h in enumerate(hints, 1):
    print(f'  Hint {i}: {h}')

### 3.3 Model B — Evaluation Metrics

Rubric §5.5: Evaluate distractor quality with P/R/F1 and train Logistic Regression
on sentence features for hint scoring.

In [ ]:
# ── Model B Evaluation ────────────────────────────────────────────────────────
# §5.5: Precision/Recall/F1 for distractor quality
# §5.3.2: ML-scored hint generation with Logistic Regression

# --- Part 1: Distractor Quality Evaluation ---
# For each test sample, generate distractors and check:
#   - Precision: fraction that are NOT the correct answer (correctly wrong)
#   - Recall: fraction that are relevant (words from article)
#   - F1: harmonic mean

dist_correct = 0   # correctly-wrong distractors
dist_total   = 0
dist_relevant = 0  # distractors from the article (relevant but wrong)

eval_rows = test_df.head(100)
for _, row in eval_rows.iterrows():
    article = str(row.get('article', ''))
    answer  = str(row.get('answer_text', ''))
    distractors = generate_distractors(article, answer, ohe_verify, top_n=3)
    for d in distractors:
        dist_total += 1
        # A good distractor is NOT the correct answer
        if clean_text(d) != clean_text(answer):
            dist_correct += 1
        # Check if distractor words appear in article (relevance)
        d_tokens = set(tokenize(d))
        a_tokens = set(tokenize(article))
        if len(d_tokens & a_tokens) > 0:
            dist_relevant += 1

dist_precision = dist_correct / max(dist_total, 1)
dist_recall    = dist_relevant / max(dist_total, 1)
dist_f1        = 2 * dist_precision * dist_recall / max(dist_precision + dist_recall, 1e-9)

print('Model B — Distractor Quality Metrics')
print(f'  Precision (not-answer): {dist_precision:.4f}')
print(f'  Recall (relevance):     {dist_recall:.4f}')
print(f'  F1 Score:               {dist_f1:.4f}')
print(f'  Total distractors evaluated: {dist_total}')

# Confusion matrix for distractor quality
# Predictions: 1 = correctly wrong, 0 = accidentally correct
dist_preds = []
dist_golds = []
for _, row in eval_rows.iterrows():
    article = str(row.get('article', ''))
    answer  = str(row.get('answer_text', ''))
    distractors = generate_distractors(article, answer, ohe_verify, top_n=3)
    for d in distractors:
        dist_preds.append(1 if clean_text(d) != clean_text(answer) else 0)
        dist_golds.append(1)  # All should be wrong (gold = 1)

print('\nDistractor Confusion Matrix:')
print(confusion_matrix(dist_golds, dist_preds))

# --- Part 2: ML-Scored Hint Evaluation (Logistic Regression) ---
# §5.3.2: Train LR on sentence features (keyword overlap, position, length)

hint_X, hint_y = [], []
for _, row in train_sample.head(200).iterrows():
    article = str(row.get('article', ''))
    question = str(row.get('question', ''))
    answer = str(row.get('answer_text', ''))
    sentences = split_sentences(article)
    if not sentences:
        continue
    feats = build_sentence_features(sentences, question, answer)
    for i, sent in enumerate(sentences):
        hint_X.append(feats[i])
        # Label: 1 if this sentence has high answer overlap
        hint_y.append(1 if sentence_keyword_overlap(sent, answer) > 0.3 else 0)

hint_X = np.array(hint_X)
hint_y = np.array(hint_y)

# Handle edge case: if only one class present
if len(set(hint_y)) < 2:
    print('\nWarning: Only one class in hint labels; skipping LR evaluation.')
else:
    from sklearn.model_selection import train_test_split
    Xh_train, Xh_test, yh_train, yh_test = train_test_split(
        hint_X, hint_y, test_size=0.3, random_state=42, stratify=hint_y)

    hint_lr = LogisticRegression(max_iter=1000, random_state=42)
    hint_lr.fit(Xh_train, yh_train)
    hint_pred = hint_lr.predict(Xh_test)

    hint_acc = accuracy_score(yh_test, hint_pred)
    hint_f1  = f1_score(yh_test, hint_pred, average='macro')
    hint_r2  = r2_score(yh_test, hint_lr.predict_proba(Xh_test)[:, 1])

    print(f'\nModel B — Hint LR Evaluation')
    print(f'  Accuracy: {hint_acc:.4f}')
    print(f'  Macro-F1: {hint_f1:.4f}')
    print(f'  R² Score: {hint_r2:.4f}')
    print()
    print(classification_report(yh_test, hint_pred,
                                target_names=['Non-Hint', 'Hint']))


---
## Phase 4 — Evaluation & Metrics Dashboard

In [ ]:
# ── Exact Match Score ────────────────────────────────────────────────────────
def exact_match(predictions, ground_truths):
    """
    Exact match: 1 if predicted answer == ground truth (after normalising).
    """
    def normalize(s):
        return clean_text(str(s), lowercase=True, remove_punct=True)
    correct = sum(normalize(p) == normalize(g) for p, g in zip(predictions, ground_truths))
    return correct / max(len(predictions), 1)


# ── Evaluate on Test set ─────────────────────────────────────────────────────
# Build test expanded dataset
exp_test = expand_to_options(test_df.sample(n=min(2000, len(test_df)), random_state=42))
Xv_test_feat = ohe_verify.transform(make_verify_corpus(exp_test))
yv_test = exp_test['is_correct'].values

test_preds, test_proba = soft_vote_predict([rf_clf, svm_clf, lr_clf], Xv_test_feat)

test_acc  = accuracy_score(yv_test, test_preds)
test_f1   = f1_score(yv_test, test_preds, average='macro')
test_prec = precision_score(yv_test, test_preds, average='macro')
test_rec  = recall_score(yv_test, test_preds, average='macro')

# R² on predicted probabilities vs ground truth
test_r2 = r2_score(yv_test, test_proba[:, 1])

# Exact match on top-predicted option per question
n_questions = len(exp_test) // 4
em_preds, em_truth = [], []
for i in range(n_questions):
    chunk_proba = test_proba[i*4:(i+1)*4, 1]
    best_idx    = np.argmax(chunk_proba)
    option_cols = [c for c in exp_test.columns if c.startswith('option')]
    pred_ans    = str(exp_test.iloc[i*4 + best_idx].get('option_text', ''))
    true_ans    = str(exp_test.iloc[i*4].get('answer_text', ''))
    em_preds.append(pred_ans)
    em_truth.append(true_ans)

em_score = exact_match(em_preds, em_truth)

print('─' * 50)
print('Model A — Test Set Metrics')
print('─' * 50)
print(f'  Accuracy (binary) : {test_acc:.4f}')
print(f'  Macro F1          : {test_f1:.4f}')
print(f'  Precision         : {test_prec:.4f}')
print(f'  Recall            : {test_rec:.4f}')
print(f'  R² Score          : {test_r2:.4f}')
print(f'  Exact Match       : {em_score:.4f}')

In [ ]:
# ── Full Metrics Dashboard Plot ───────────────────────────────────────────────
fig = plt.figure(figsize=(16, 12))

# 1 — Model comparison bar chart
ax1 = fig.add_subplot(2, 3, 1)
metrics_df = pd.DataFrame({
    'Model':    ['RF', 'SVM', 'LR', 'Ensemble'],
    'F1':       [rf_f1, svm_f1, lr_f1, ens_f1],
    'Accuracy': [rf_acc, svm_acc, lr_acc, ens_acc]
})
x = np.arange(len(metrics_df))
w = 0.35
ax1.bar(x - w/2, metrics_df['Accuracy'], w, label='Accuracy', color='#4C78A8')
ax1.bar(x + w/2, metrics_df['F1'],       w, label='Macro-F1', color='#F58518')
ax1.set_xticks(x); ax1.set_xticklabels(metrics_df['Model'])
ax1.set_ylim(0, 1.05); ax1.set_title('Model A — Dev Performance')
ax1.legend(fontsize=8)

# 2 — Confusion matrix
ax2 = fig.add_subplot(2, 3, 2)
cm  = confusion_matrix(yv_test, test_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax2,
            xticklabels=['Incorrect','Correct'], yticklabels=['Incorrect','Correct'])
ax2.set_title('Test Confusion Matrix'); ax2.set_xlabel('Predicted'); ax2.set_ylabel('Actual')

# 3 — Cluster analysis
ax3 = fig.add_subplot(2, 3, 3)
ax3.bar(cluster_summary.index, cluster_summary['correct_rate'],
        color='#54A24B', edgecolor='white')
ax3.axhline(0.25, color='red', ls='--', lw=1.5, label='Baseline')
ax3.set_title('K-Means Cluster Correct Rates')
ax3.set_xlabel('Cluster'); ax3.set_ylabel('Correct Rate'); ax3.legend(fontsize=8)

# 4 — ROC-like probability histogram
ax4 = fig.add_subplot(2, 3, 4)
ax4.hist(test_proba[yv_test==0, 1], bins=30, alpha=0.6, color='#E45756', label='Incorrect')
ax4.hist(test_proba[yv_test==1, 1], bins=30, alpha=0.6, color='#4C78A8', label='Correct')
ax4.set_title('Predicted P(Correct) Distribution')
ax4.set_xlabel('P(Correct)'); ax4.legend(fontsize=8)

# 5 — Metrics table
ax5 = fig.add_subplot(2, 3, 5)
ax5.axis('off')
table_data = [
    ['Metric', 'Dev', 'Test'],
    ['Accuracy', f'{ens_acc:.4f}', f'{test_acc:.4f}'],
    ['Macro-F1', f'{ens_f1:.4f}',  f'{test_f1:.4f}'],
    ['Precision',f'{ens_prec:.4f}',f'{test_prec:.4f}'],
    ['Recall',   f'{ens_rec:.4f}', f'{test_rec:.4f}'],
    ['R²',       '—',             f'{test_r2:.4f}'],
    ['Exact Match','—',           f'{em_score:.4f}'],
]
tbl = ax5.table(cellText=table_data, loc='center', cellLoc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(9)
tbl.scale(1.2, 1.6)
ax5.set_title('Summary Metrics', pad=20)

# 6 — Answer length distribution
ax6 = fig.add_subplot(2, 3, 6)
train_df['answer_len'].hist(bins=30, ax=ax6, color='#72B7B2', edgecolor='white')
ax6.set_title('Answer Length Distribution')
ax6.set_xlabel('Word Count'); ax6.set_ylabel('Freq')

plt.suptitle('RACE ML Pipeline — Analytics Dashboard', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('content/analytics_dashboard.png', dpi=130, bbox_inches='tight')
plt.show()
print('Dashboard saved to content/analytics_dashboard.png')